In [4]:
import os
import json
import time
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import numpy as np

load_dotenv()
client = OpenAI()

In [5]:
df = pd.read_excel("RAG_qa.xlsx")
df = df[["question", "answer"]].copy()
print(f"총 평가 항목: {len(df)}개")
df.head(3)

총 평가 항목: 697개


,question,answer
0,A형간염 알아보기 작성에 사용한 참고자료를 알려줘,참고자료로 사용된 A형간염 관련 정보는 부산광역시 A형간염 관리 지침과 질병관리청의...
1,A형간염 예방접종은?,"A형간염 예방접종은 만 40세 미만 출생자는 항체검사 없이 백신 접종하고, 만 40..."
2,A형간염의 임상증상은?,"A형간염의 임상증상은 발열, 두통, 권태감, 식욕부진, 오심, 구토, 복통, 설사 ..."


In [6]:
JUDGE_PROMPT = """
당신은 법정감염병 챗봇의 답변 품질을 평가하는 전문가입니다.
아래 질문과 답변을 읽고, 다음 3가지 기준으로 각각 1~5점을 부여하세요.

[평가 기준]
- accuracy (정확성): 감염병 관련 사실 정보가 정확하고 오류가 없는가 (1=명백한 오류 있음, 5=완전히 정확)
- relevance (관련성): 답변이 질문에 직접 대답하는가 (1=전혀 관련 없음, 5=완벽히 일치)
- completeness (완전성): 질문이 필요로 하는 정보를 충분히 포함하는가 (1=매우 불충분, 5=완전히 충분)

[출력 형식] JSON만 출력하세요:
{"accuracy": <1-5>, "relevance": <1-5>, "completeness": <1-5>, "reason": "<간단한 근거>"}

[질문]: {question}
[답변]: {answer}
"""

def judge(question: str, answer: str, retries: int = 3) -> dict:
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model="gpt-4o",
                messages=[
                    {"role": "user", "content": JUDGE_PROMPT.format(
                        question=question, answer=answer)}
                ],
                temperature=0,
                response_format={"type": "json_object"}
            )
            return json.loads(resp.choices[0].message.content)
        except Exception as e:
            if attempt == retries - 1:
                return {"accuracy": None, "relevance": None, "completeness": None, "reason": str(e)}
            time.sleep(2 ** attempt)

In [7]:
SAMPLE_SIZE = None  # None이면 전체, 숫자 지정 시 샘플링 (테스트 시 5 또는 100 권장)

eval_df = df.sample(SAMPLE_SIZE, random_state=42) if SAMPLE_SIZE else df.copy()
results = []

for i, row in eval_df.iterrows():
    scores = judge(row["question"], row["answer"])
    scores["question"] = row["question"]
    scores["answer"] = row["answer"]
    results.append(scores)

    if (len(results) % 50) == 0:
        pd.DataFrame(results).to_csv("eval_results_partial.csv", index=False)
        print(f"[{len(results)}/{len(eval_df)}] 중간 저장 완료")

    time.sleep(0.5)

result_df = pd.DataFrame(results)
result_df["total_score"] = result_df[["accuracy", "relevance", "completeness"]].mean(axis=1)
result_df.to_csv("eval_results.csv", index=False)
print("평가 완료!")
result_df.head()

[50/697] 중간 저장 완료


KeyboardInterrupt: 

In [ ]:
metrics = ["accuracy", "relevance", "completeness", "total_score"]
summary = result_df[metrics].describe().loc[["mean", "std", "min", "50%", "max"]]
print(summary.round(3))

NameError: name 'result_df' is not defined

In [ ]:
# 한글 폰트 설정 (macOS)
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

metrics = ["accuracy", "relevance", "completeness", "total_score"]
labels = ["정확성", "관련성", "완전성", "종합점수"]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col, label in zip(axes.flatten(), metrics, labels):
    ax.hist(result_df[col].dropna(), bins=5, range=(1, 5), edgecolor="black", color="steelblue", alpha=0.8)
    ax.set_title(f"{label} ({col}) 분포", fontsize=13)
    ax.set_xlabel("Score (1~5)")
    ax.set_ylabel("Count")
    ax.axvline(result_df[col].mean(), color="red", linestyle="--", label=f"평균: {result_df[col].mean():.2f}")
    ax.legend()

plt.suptitle("법정감염병 챗봇 LLM-as-Judge 평가 결과", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig("eval_distribution.png", dpi=150)
plt.show()

# 기준별 평균 막대 차트
fig2, ax2 = plt.subplots(figsize=(7, 5))
means = result_df[["accuracy", "relevance", "completeness"]].mean()
bars = ax2.bar(labels[:3], means.values, color=["#4C72B0", "#DD8452", "#55A868"], edgecolor="black")
ax2.set_ylim(0, 5)
ax2.set_ylabel("평균 점수")
ax2.set_title("평가 기준별 평균 점수", fontsize=13)
for bar, val in zip(bars, means.values):
    ax2.text(bar.get_x() + bar.get_width() / 2, val + 0.05, f"{val:.2f}", ha="center", fontsize=11)
plt.tight_layout()
plt.savefig("eval_bar_chart.png", dpi=150)
plt.show()

NameError: name 'plt' is not defined

In [ ]:
worst = result_df.nsmallest(10, "total_score")[["question", "total_score", "reason"]]
print(worst.to_string())